# Process Monitor Insert - Standalone

This notebook provides a standalone function to insert entries into the `process_monitor_logs` table. It is designed to be shareable with team members and includes all necessary database connection logic.

## How to Use

1. Modify the `DB_PARAMS` dictionary with your database connection details
2. Use the `insert_process_monitor_entry` function to add new entries to the process monitoring table
3. See the example usage in the second cell

In [ ]:
import json
import uuid
import logging
import datetime
from typing import Dict, Any, List, Optional, Union
import psycopg2
from psycopg2.extras import register_uuid, Json

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Register UUID adapter for PostgreSQL
register_uuid()

# ------------------------------------------------------
# DATABASE CONFIGURATION - MODIFY THIS SECTION
# ------------------------------------------------------
# Replace these values with your own database credentials
DB_PARAMS = {
    "host": "localhost",
    "port": "5432",
    "dbname": "maven-finance",
    "user": "iris_dev",
    "password": "",  # Add password if required
}
# ------------------------------------------------------

def connect_to_db():
    """
    Connect to the PostgreSQL database using the configured parameters.
    
    Returns:
        Database connection object or None if connection fails
    """
    try:
        logger.info(f"Connecting to database with parameters: host={DB_PARAMS['host']}, "
                    f"port={DB_PARAMS['port']}, dbname={DB_PARAMS['dbname']}, user={DB_PARAMS['user']}")
        conn = psycopg2.connect(**DB_PARAMS)
        conn.autocommit = False
        logger.info("Database connection successful")
        return conn
    except Exception as e:
        logger.error(f"Error connecting to database: {e}")
        return None

def insert_process_monitor_entry(
    # Required fields
    run_uuid: Union[uuid.UUID, str],
    model_name: str,
    stage_name: str,
    stage_start_time: datetime.datetime,
    # Optional fields with sensible defaults
    stage_end_time: Optional[datetime.datetime] = None,
    duration_ms: Optional[int] = None,
    llm_calls: Optional[List[Dict[str, Any]]] = None,
    total_tokens: Optional[int] = None,
    total_cost: Optional[float] = None,
    status: Optional[str] = None,
    decision_details: Optional[str] = None,
    error_message: Optional[str] = None,
    user_id: Optional[str] = None,
    environment: Optional[str] = None,
    custom_metadata: Optional[Dict[str, Any]] = None,
    notes: Optional[str] = None
) -> bool:
    """
    Insert a new entry into the process_monitor_logs table.
    
    Args:
        run_uuid: Unique ID for the entire model run (can be string or UUID object)
        model_name: Name of the model being used
        stage_name: Name of the process stage
        stage_start_time: When the stage began
        stage_end_time: When the stage ended (optional)
        duration_ms: Duration in milliseconds (optional, calculated if both start/end times provided)
        llm_calls: List of LLM call details as dictionaries (optional)
        total_tokens: Total token count (optional, calculated from llm_calls if provided)
        total_cost: Total cost (optional, calculated from llm_calls if provided)
        status: Status of the stage (e.g., 'Success', 'Failure')
        decision_details: Details about decisions made during the stage
        error_message: Error message if the stage failed
        user_id: ID of the user who initiated the request
        environment: Environment name (e.g., 'production', 'development')
        custom_metadata: Any additional structured metadata as a dictionary
        notes: Additional free-form notes
        
    Returns:
        bool: True if insertion was successful, False otherwise
    """
    # Validations for required fields
    if not run_uuid or not model_name or not stage_name or not stage_start_time:
        logger.error("Missing required fields: run_uuid, model_name, stage_name, and stage_start_time are required")
        return False
    
    # Convert string UUID to UUID object if needed
    if isinstance(run_uuid, str):
        try:
            run_uuid = uuid.UUID(run_uuid)
        except ValueError as e:
            logger.error(f"Invalid UUID format: {e}")
            return False
    
    # Calculate duration if not provided but have start and end times
    if duration_ms is None and stage_end_time is not None:
        try:
            duration_ms = int((stage_end_time - stage_start_time).total_seconds() * 1000)
        except Exception as e:
            logger.warning(f"Could not calculate duration: {e}. Will insert NULL for duration_ms.")
    
    # Process llm_calls if provided
    if llm_calls is not None:
        # Calculate totals if not provided
        if total_tokens is None:
            total_tokens = sum(call.get('input_tokens', 0) + call.get('output_tokens', 0) 
                              for call in llm_calls if isinstance(call, dict))
        
        if total_cost is None:
            total_cost = sum(call.get('cost', 0) 
                            for call in llm_calls if isinstance(call, dict))
    
    # Connect to the database
    conn = connect_to_db()
    if not conn:
        logger.error("Failed to connect to database. Cannot insert monitor entry.")
        return False
    
    # Insert the entry
    try:
        with conn.cursor() as cur:
            query = """
            INSERT INTO process_monitor_logs (
                run_uuid, model_name, stage_name, stage_start_time, stage_end_time,
                duration_ms, llm_calls, total_tokens, total_cost, status,
                decision_details, error_message, user_id, environment, custom_metadata, notes
            ) VALUES (
                %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s
            ) RETURNING log_id
            """
            
            # Prepare llm_calls and custom_metadata as JSONB
            llm_calls_jsonb = Json(llm_calls) if llm_calls is not None else None
            custom_metadata_jsonb = Json(custom_metadata) if custom_metadata is not None else None
            
            cur.execute(query, (
                run_uuid, model_name, stage_name, stage_start_time, stage_end_time,
                duration_ms, llm_calls_jsonb, total_tokens, total_cost, status,
                decision_details, error_message, user_id, environment, custom_metadata_jsonb, notes
            ))
            
            # Get the inserted log_id
            result = cur.fetchone()
            log_id = result[0] if result else None
            
            # Commit the transaction
            conn.commit()
            
            logger.info(f"Successfully inserted process monitor entry with log_id: {log_id}")
            return True
            
    except Exception as e:
        conn.rollback()
        logger.error(f"Error inserting process monitor entry: {e}")
        return False
    finally:
        conn.close()

# Function to verify the table exists (helpful for first-time users)
def verify_process_monitor_table_exists() -> bool:
    """
    Check if the process_monitor_logs table exists in the database.
    
    Returns:
        bool: True if the table exists, False otherwise
    """
    conn = connect_to_db()
    if not conn:
        return False
    
    try:
        with conn.cursor() as cur:
            cur.execute("""
                SELECT EXISTS (
                    SELECT FROM information_schema.tables 
                    WHERE table_schema = 'public'
                    AND table_name = 'process_monitor_logs'
                );
            """)
            return cur.fetchone()[0]
    except Exception as e:
        logger.error(f"Error checking if table exists: {e}")
        return False
    finally:
        conn.close()

# Check if the table exists when this cell is run
table_exists = verify_process_monitor_table_exists()
if table_exists:
    logger.info("✅ process_monitor_logs table exists in the database!")
else:
    logger.warning("⚠️ process_monitor_logs table doesn't exist in the database!")
    logger.info("If needed, run the SQL from postgres_schema.sql to create the table.")

In [ ]:
# Example usage of the process_monitor_entry function

# Generate a mock run UUID (you would typically use a consistent UUID across related stages)
mock_run_uuid = str(uuid.uuid4())
print(f"Using mock run UUID: {mock_run_uuid}")

# Example LLM calls data
mock_llm_calls = [
    {
        "model": "gpt-4",
        "input_tokens": 320,
        "output_tokens": 150,
        "cost": 0.0145,
        "response_time_ms": 3200
    }
]

# Custom metadata example
mock_custom_metadata = {
    "session_id": "abc123",
    "priority": "high",
    "test_mode": True
}

# Create mock stage timestamps
now = datetime.datetime.now(datetime.timezone.utc)
mock_start_time = now - datetime.timedelta(seconds=5)  # 5 seconds ago
mock_end_time = now  # Now

# Insert a test entry
success = insert_process_monitor_entry(
    # Required fields
    run_uuid=mock_run_uuid,
    model_name="test_model",
    stage_name="test_process",
    stage_start_time=mock_start_time,
    
    # Optional fields
    stage_end_time=mock_end_time,
    duration_ms=5000,  # 5 seconds in ms
    llm_calls=mock_llm_calls,
    total_tokens=470,  # input + output tokens
    total_cost=0.0145,
    status="Success",
    decision_details="Test process completed successfully",
    error_message=None,
    user_id="test_user",
    environment="development",
    custom_metadata=mock_custom_metadata,
    notes="This is a test entry from the standalone notebook"
)

if success:
    print("✅ Test entry inserted successfully!")
else:
    print("❌ Failed to insert test entry.")

# Example: Insert a minimalist entry with only required fields
minimal_success = insert_process_monitor_entry(
    run_uuid=mock_run_uuid,
    model_name="test_model",
    stage_name="minimal_test",
    stage_start_time=datetime.datetime.now(datetime.timezone.utc)
)

if minimal_success:
    print("✅ Minimal test entry inserted successfully!")
else:
    print("❌ Failed to insert minimal test entry.")